# Hypothesis Testing

This notebook applies **inferential statistical techniques** to evaluate business hypotheses derived from exploratory data analysis.

The goal is to determine whether observed patterns in the dataset are **statistically significant or occurred by chance**.

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
df = pd.read_csv('/content/drive/MyDrive/olist-end-to-end-analytics/data/olist_master_orders_features.csv')

In [5]:
df.head()

,order_id,customer_id,order_purchase_timestamp,order_date,delivery_delay_days,delivery_status,review_score,order_value,freight_value,total_order_value,payment_type,payment_installments,payment_value,late_delivery_flag,high_value_order,order_month
0,6e864b3f0ec71031117ad4cf46b7f2a1,9f9d249355f63c5c1216a82b802452c1,2018-04-24 20:15:21 UTC,2018-04-24,-14.0,On Time,5.0,0.85,18.23,19.08,credit_card,1.0,19.08,0,0,4
1,3ee6513ae7ea23bdfab5b9ab60bffcb5,161b6d415e8b3413c6609c70cf405b5a,2018-04-24 11:01:06 UTC,2018-04-24,-10.0,On Time,4.0,0.85,18.23,19.08,boleto,1.0,19.08,0,0,4
2,f1d5c2e6867fa93ceee9ef9b34a53cbf,a790343ca6f3fee08112d678b43aa7c5,2018-08-25 21:20:50 UTC,2018-08-25,-5.0,On Time,5.0,2.20,7.39,9.59,credit_card,1.0,0.31,0,0,8
3,f1d5c2e6867fa93ceee9ef9b34a53cbf,a790343ca6f3fee08112d678b43aa7c5,2018-08-25 21:20:50 UTC,2018-08-25,-5.0,On Time,5.0,2.20,7.39,9.59,voucher,1.0,9.28,0,0,8
4,e8bbc1d69fee39eee4c72cb5c969e39d,184e8e8e48937145eb96c721ef1f0747,2017-09-13 19:13:20 UTC,2017-09-13,-8.0,On Time,5.0,2.29,7.78,10.07,credit_card,1.0,10.07,0,0,9


## Hypothesis Test 1: Impact of Delivery Delays on Customer Review Scores

Delivery performance is a critical factor in customer satisfaction for e-commerce platforms. Delays in delivery may negatively affect customer experience and result in lower review scores.

In this analysis, we test whether **late deliveries significantly reduce customer review ratings**.

### Hypotheses

**Null Hypothesis (H₀):**
There is **no difference** in the average review score between on-time deliveries and late deliveries.

**Alternative Hypothesis (H₁):**
Late deliveries have **lower average review scores** compared to on-time deliveries.

### Statistical Test

To evaluate this, we perform an **Independent Two-Sample t-test**, which compares the means of two independent groups:

* **Group 1:** On-time deliveries
* **Group 2:** Late deliveries

If the **p-value < 0.05**, we reject the null hypothesis and conclude that delivery delays significantly affect customer review scores.

In [6]:
# creating two groups
on_time = df[df['late_delivery_flag'] == 0]['review_score'].dropna()
late = df[df['late_delivery_flag'] == 1]['review_score'].dropna()

print('On-time deliveries:',len(on_time))
print('Late deliveries:',len(late))

On-time deliveries: 93997
Late deliveries: 6653


In [7]:
t_stat,p_value = stats.ttest_ind(on_time,late)

print('T-statistic:',t_stat)
print('P-value:',p_value)

T-statistic: 134.56705561981278
P-value: 0.0


In [8]:
print('Average review score (On-time):',on_time.mean())
print('Average review score (Late):',late.mean())

Average review score (On-time): 4.2876368394736
Average review score (Late): 2.266346009319104


In [9]:
mean_diff = on_time.mean() - late.mean()

ci = stats.t.interval(
    0.95,
    len(on_time)-1,
    loc=mean_diff,
    scale=stats.sem(on_time)
)
print('Mean Difference:',mean_diff)
print('95% Confidence Interval:',ci)

Mean Difference: 2.021290830154496
95% Confidence Interval: (np.float64(2.013926475269582), np.float64(2.0286551850394097))


### Hypothesis Test Results

The analysis compared customer review scores between **on-time deliveries** and **late deliveries**.

* Number of on-time deliveries: **93,997**
* Number of late deliveries: **6,653**

The **average review score** for on-time deliveries is **4.29**, while the average review score for late deliveries is **2.27**.

The **t-statistic is 134.57** and the **p-value is effectively 0**, which is far below the significance threshold of **0.05**.

### Statistical Conclusion

Since the **p-value < 0.05**, we **reject the null hypothesis**.
This indicates that there is a **statistically significant difference in review scores between on-time and late deliveries**.

### Business Interpretation

The results show that **late deliveries significantly reduce customer satisfaction**.
On average, late deliveries receive review scores **about 2 points lower** than on-time deliveries.

The **95% confidence interval (2.01 to 2.03)** suggests that the true difference in review scores between the two groups is consistently around **2 points**, reinforcing the strength of this relationship.

### Key Insight

Delivery performance is a **major driver of customer satisfaction**. Improving logistics efficiency and reducing delivery delays could significantly improve customer ratings and overall platform reputation.

## Hypothesis Test 2: Do Credit Card Users Spend More Than Boleto Users?

Different payment methods may reflect different purchasing behaviors. Customers who use credit cards may be more willing to make larger purchases compared to those using boleto payments.

In this analysis, we test whether the **average order value differs between credit card payments and boleto payments**.

### Hypotheses

**Null Hypothesis (H₀):**
There is **no difference in the average order value** between credit card and boleto payments.

**Alternative Hypothesis (H₁):**
The **average order value differs between credit card and boleto payments**.

### Statistical Test

We apply an **Independent Two-Sample t-test** to compare the mean order values between:

* **Group 1:** Credit card payments
* **Group 2:** Boleto payments

If the **p-value < 0.05**, we reject the null hypothesis and conclude that the payment method significantly affects order value.

In [14]:
credit_card = df[df['payment_type'] == 'credit_card']['total_order_value'].dropna()
boleto = df[df['payment_type'] == 'boleto']['total_order_value'].dropna()

print('Credit card transactions:',len(credit_card))
print('Boleto transactions:',len(boleto))

Credit card transactions: 74976
Boleto transactions: 19308


In [11]:
t_stat,p_value = stats.ttest_ind(credit_card,boleto)

print('T-statistic:',t_stat)
print('P-value:',p_value)

T-statistic: 12.075206113557899
P-value: 1.5119559745704109e-33


In [12]:
print('Average order value (Credit Card):',credit_card.mean())
print('Average order value (Boleto):',boleto.mean())

Average order value (Credit Card): 165.81471804310712
Average order value (Boleto): 144.31218458669983


In [13]:
mean_diff2 = credit_card.mean() - boleto.mean()

ci2 = stats.t.interval(
    0.95,
    len(credit_card)-1,
    loc = mean_diff2,
    scale = stats.sem(credit_card)
)

print('Mean Difference:',mean_diff2)
print('95% Confidence Interval:',ci2)

Mean Difference: 21.502533456407292
95% Confidence Interval: (np.float64(19.90866764519524), np.float64(23.096399267619343))


### Hypothesis Test Results

This test compared the **average order value between credit card payments and boleto payments**.

* Number of credit card transactions: **74,976**
* Number of boleto transactions: **19,308**

The **average order value for credit card payments is 165.81 BRL**, while the **average order value for boleto payments is 144.31 BRL**.

The **t-statistic is 12.08** and the **p-value is extremely small (1.51 × 10⁻³³)**, which is far below the significance level of **0.05**.

### Statistical Conclusion

Since the **p-value < 0.05**, we **reject the null hypothesis**.
This indicates that there is a **statistically significant difference in order values between credit card and boleto payments**.

### Business Interpretation

Customers who pay using **credit cards tend to spend more on average than those using boleto payments**.

The **average difference in spending is about 21.5 BRL**, meaning credit card users typically place **higher-value orders**.

The **95% confidence interval (19.91 to 23.10)** suggests that the true difference in average order value between the two payment methods is consistently around **20–23 BRL**.

### Key Insight

Payment method appears to influence purchasing behavior.
Encouraging **credit card payments through promotions or installment options** may increase the **average order value and overall revenue** for the platform.

## Hypothesis Test 3: Do Different Payment Methods Lead to Different Average Order Values?

Customers use multiple payment methods on the platform, including **credit cards, boleto, vouchers, and debit cards**. These payment options may reflect different purchasing behaviors.

In this analysis, we examine whether the **average order value differs across different payment methods**.

### Hypotheses

**Null Hypothesis (H₀):**
The **average order value is the same across all payment methods**.

**Alternative Hypothesis (H₁):**
At least **one payment method has a different average order value** compared to the others.

### Statistical Test

Since we are comparing **more than two groups**, we use a **One-Way ANOVA (Analysis of Variance)** test.

The test compares the average order value across the following payment types:

* Credit Card
* Boleto
* Voucher
* Debit Card

If the **p-value < 0.05**, we reject the null hypothesis and conclude that **payment method significantly affects order value**.

In [15]:
df['payment_type'].value_counts()

,count
payment_type,
credit_card,74976
boleto,19308
voucher,5548
debit_card,1493


In [16]:
credit_card = df[df['payment_type'] == 'credit_card']['total_order_value'].dropna()
boleto = df[df['payment_type'] == 'boleto']['total_order_value'].dropna()
voucher = df[df['payment_type'] == 'voucher']['total_order_value'].dropna()
debit_card = df[df['payment_type'] == 'debit_card']['total_order_value'].dropna()

In [17]:
f_stat, p_value = stats.f_oneway(credit_card,boleto,voucher,debit_card)

print('F-statistics:',f_stat)
print('P-value:',p_value)

F-statistics: 75.11537730958729
P-value: 1.5879794984224966e-48


In [18]:
print("Average order value (Credit Card):", credit_card.mean())
print("Average order value (Boleto):", boleto.mean())
print("Average order value (Voucher):", voucher.mean())
print("Average order value (Debit Card):", debit_card.mean())

Average order value (Credit Card): 165.81471804310712
Average order value (Boleto): 144.31218458669983
Average order value (Voucher): 137.12659336697908
Average order value (Debit Card): 140.2233556597455


### Hypothesis Test Results

This test examined whether the **average order value differs across payment methods**.

Number of transactions by payment type:

* Credit Card: **74,976**
* Boleto: **19,308**
* Voucher: **5,548**
* Debit Card: **1,493**

The **ANOVA test produced an F-statistic of 75.12 and a p-value of 1.59 × 10⁻⁴⁸**, which is far below the significance level of **0.05**.

### Statistical Conclusion

Since the **p-value < 0.05**, we **reject the null hypothesis**.
This indicates that **at least one payment method has a significantly different average order value compared to the others**.

### Average Order Value by Payment Method

* Credit Card: **165.81 BRL**
* Boleto: **144.31 BRL**
* Voucher: **137.13 BRL**
* Debit Card: **140.22 BRL**

### Business Interpretation

The results suggest that **payment method is associated with different spending behaviors**.

Customers paying with **credit cards tend to make higher-value purchases on average**, while orders paid through **vouchers and debit cards tend to have lower average values**.

### Key Insight

Encouraging payment methods that are associated with **higher spending, such as credit cards**, could potentially increase the **average order value and overall revenue** for the platform.

In [20]:
corr, p_value = stats.pearsonr(df['total_order_value'], df['freight_value'])

print("Correlation:", corr)
print("P-value:", p_value)

Correlation: 0.49295887760952206
P-value: 0.0


### Correlation Analysis: Order Value vs Freight Cost

To understand the relationship between product price and shipping cost, we examined the correlation between **total order value** and **freight value**.

The **Pearson correlation coefficient is 0.49**, indicating a **moderate positive relationship** between order value and shipping cost.

The **p-value is effectively 0**, which means the relationship is **statistically significant**.

### Interpretation

This result suggests that **larger orders tend to have higher shipping costs**. This relationship is expected because higher-value purchases may involve:

* Larger or heavier items
* Multiple products in one order
* Products shipped from more distant locations

### Business Insight

Shipping costs appear to scale with order size, meaning logistics expenses increase as purchase value grows. Understanding this relationship can help the platform **optimize shipping strategies and pricing policies**.


## Final Insights

This analysis applied inferential statistics to examine key relationships in the e-commerce dataset.

Several important findings emerged from the hypothesis tests:

**1. Delivery delays significantly reduce customer satisfaction.**
Orders delivered late receive review scores that are approximately **2 points lower on average** than on-time deliveries, indicating that logistics performance strongly influences customer experience.

**2. Payment method influences customer spending behavior.**
Customers paying with **credit cards tend to place higher-value orders** compared to those using boleto payments.

**3. Order value differs across payment methods.**
The ANOVA test confirmed that **average order value varies significantly between payment types**, with credit card users generally spending more.

**4. Shipping cost increases with order value.**
The correlation analysis showed a **moderate positive relationship between order value and freight cost**, indicating that larger purchases are associated with higher shipping expenses.

### Overall Conclusion

These findings highlight several important drivers of performance in the e-commerce platform:

* **Efficient delivery operations are critical for maintaining high customer satisfaction.**
* **Payment behavior influences purchasing patterns and revenue generation.**
* **Shipping costs scale with order size, which may impact logistics planning and pricing strategies.**

Understanding these relationships can help businesses **improve operational efficiency, enhance customer satisfaction, and optimize revenue strategies**.
